In [3]:
import numpy as np
from collections import defaultdict

class SimpleBlackjackEnv:
    def __init__(self):
        pass

    def _get_random_state(self):
        player_sum = np.random.randint(12, 22)
        dealer_card = np.random.randint(1, 11)
        usable_ace = np.random.choice([True, False])
        return (player_sum, dealer_card, usable_ace)

    def _get_random_action(self):
        return np.random.choice([0, 1])

    def simulate_episode(self, policy=None):
        episode = []
        state = self._get_random_state()
        is_done = False
        t = 0
        while not is_done and t < 10:
            if policy:
                action = policy(state)
            else:
                action = self._get_random_action()

            reward = np.random.choice([-1, 0, 1])
            next_state = self._get_random_state()
            is_done = np.random.rand() < 0.3

            episode.append((state, action, reward))
            state = next_state
            t += 1

        if is_done:
             episode.append((state, None, np.random.choice([-1, 0, 1])))

        return episode

def first_visit_mc_prediction(env, num_episodes, discount_factor=1.0):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)
    V = defaultdict(float)

    for i_episode in range(1, num_episodes + 1):
        episode = env.simulate_episode()
        rewards_in_episode = [x[2] for x in episode]

        states_visited = set()

        for t, (state, action, reward) in enumerate(episode):
            if state not in states_visited:
                states_visited.add(state)

                G = 0
                for k in range(t, len(episode)):
                    G += (discount_factor**(k - t)) * rewards_in_episode[k]

                returns_sum[state] += G
                returns_count[state] += 1
                V[state] = returns_sum[state] / returns_count[state]

    return V

env = SimpleBlackjackEnv()
num_prediction_episodes = 10000

V_values = first_visit_mc_prediction(env, num_prediction_episodes)

In [6]:
print(f"Estimated V values after {num_prediction_episodes} episodes:")

count = 0
for state, value in sorted(V_values.items()):
    if count < 5:
        print(f"V({state}) = {value:.2f}")
        count += 1
    else:
        break
print("(Displaying first 5 states for brevity)")

Estimated V values after 10000 episodes:
V((12, 1, np.False_)) = -0.05
V((12, 1, np.True_)) = -0.13
V((12, 2, np.False_)) = -0.05
V((12, 2, np.True_)) = -0.05
V((12, 3, np.False_)) = 0.19
(Displaying first 5 states for brevity)


In [7]:
import numpy as np
from collections import defaultdict



def make_epsilon_greedy_policy(Q, epsilon, nA):
    def policy_fn(state):
        action_probs = np.ones(nA, dtype=float) * epsilon / nA
        best_action = np.argmax(Q[state])
        action_probs[best_action] += (1.0 - epsilon)
        return np.random.choice(np.arange(nA), p=action_probs)
    return policy_fn

def on_policy_first_visit_mc_control(env, num_episodes, discount_factor=1.0, epsilon=0.1):
    nA = 2

    sa_returns_sum = defaultdict(float)
    sa_returns_count = defaultdict(int)
    Q = defaultdict(lambda: np.zeros(nA))

    policy = make_epsilon_greedy_policy(Q, epsilon, nA)

    for i_episode in range(1, num_episodes + 1):
        episode = env.simulate_episode(policy)
        rewards_in_episode = [x[2] for x in episode]

        sa_visited = set()

        for t, (state, action, reward) in enumerate(episode):
            if action is None:
                continue

            state_action_pair = (state, action)

            if state_action_pair not in sa_visited:
                sa_visited.add(state_action_pair)

                G = 0
                for k in range(t, len(episode)):
                    G += (discount_factor**(k - t)) * rewards_in_episode[k]

                sa_returns_sum[state_action_pair] += G
                sa_returns_count[state_action_pair] += 1
                Q[state][action] = sa_returns_sum[state_action_pair] / sa_returns_count[state_action_pair]

        policy = make_epsilon_greedy_policy(Q, epsilon, nA)

    return Q, policy

env_mc = SimpleBlackjackEnv()
num_control_episodes = 50000

Q_values, optimal_policy = on_policy_first_visit_mc_control(env_mc, num_control_episodes, epsilon=0.2)

In [8]:
print(f"Estimated Q values and optimal policy after {num_control_episodes} episodes:")
# Display a few estimated Q values and corresponding optimal actions
count = 0
for state, q_actions in sorted(Q_values.items()):
    if count < 5:
        optimal_action = np.argmax(q_actions)
        print(f"Q({state}) = {q_actions[0]:.2f} (Stand), {q_actions[1]:.2f} (Hit) | Optimal Action: {'Hit' if optimal_action == 1 else 'Stand'}")
        count += 1
    else:
        break
print("(Displaying first 5 states for brevity)")

Estimated Q values and optimal policy after 50000 episodes:
Q((12, 1, np.False_)) = 0.14 (Stand), -0.16 (Hit) | Optimal Action: Stand
Q((12, 1, np.True_)) = 0.03 (Stand), 0.07 (Hit) | Optimal Action: Hit
Q((12, 2, np.False_)) = 0.09 (Stand), -0.02 (Hit) | Optimal Action: Stand
Q((12, 2, np.True_)) = 0.01 (Stand), 0.06 (Hit) | Optimal Action: Hit
Q((12, 3, np.False_)) = 0.16 (Stand), 0.13 (Hit) | Optimal Action: Stand
(Displaying first 5 states for brevity)
